# Train the CANINE re-ranker

This notebook trains a second Stage 2 re-ranker on CANINE (google/canine-c), a transformer that reads raw characters instead of word pieces. It compares the CANINE re-ranker with the MiniLM re-ranker from 4_train_reranker.ipynb. Both use the same training pairs and the same retriever candidates. Only the backbone and the input limit change: 144 characters, against the MiniLM re-ranker's 64 word pieces. The evaluation follows 4_train_reranker.ipynb on the 950 test-set queries, with McNemar's exact test against the retriever alone and against the MiniLM re-ranker.

- Reads: the retriever's model, per-query files and cached candidates; the MiniLM re-ranker's mined arrays, manifest, model and saved predictions; the synthetic training set; the test set; gnaf.duckdb (read-only).
- Writes: output_canine/ (scratch/canine_smoke/ with C_SMOKE=1).
- Run after: 4_train_reranker.ipynb

Training (cell 6) takes about an hour and scoring every candidate with CANINE (cell 8) about half an hour. With C_ALLOW_CACHE=1 the saved model is loaded instead of trained.

In [ ]:
# Cell 1: configuration. Every input is read-only and every output goes to OUT.
# The hyperparameters are those of 4_train_reranker.ipynb cell 2, except the
# backbone and MAX_CHARS. The mining quotas are restated only to rebuild the
# pairs. Nothing is mined here.
# Switches: C_SMOKE=1 runs 30 training steps, scores K<=10 and writes to
# scratch/canine_smoke/. C_ALLOW_CACHE=1 loads OUT/model instead of training.
# C_OUT, C_MODEL, S1_OUT, S2_OUT, C_MISS, C_DB, C_SYN and C_S2PRED override paths.

import os, sys, json, time, re, hashlib, zlib, math
from pathlib import Path
import numpy as np, pandas as pd

def _env(name, default): return os.environ.get(name, default)

S1_OUT   = Path(_env("S1_OUT", "output_stage1"))
S1_MODEL = S1_OUT / "model"
S1_PERQ  = S1_OUT / "stage1_recall_per_query.tsv"
S1_SUB   = S1_OUT / "stage1_recall_by_subset.tsv"
S2_OUT   = Path(_env("S2_OUT", "output_stage2"))
S2_MODEL = S2_OUT / "model"                       # the MiniLM re-ranker
MP, MS, MSIDE = S2_OUT / "mine_pids.npy", S2_OUT / "mine_sims.npy", S2_OUT / "mine_progress.json"
EP, ES, ESIDE = S2_OUT / "eval_pids.npy", S2_OUT / "eval_sims.npy", S2_OUT / "eval_retrieval.json"
S2_MANIF = S2_OUT / "s2_manifest.json"
S2_PRED  = Path(_env("C_S2PRED", "results_labels_789_model_refit/stage2/stage2_predictions.tsv"))
S2_E2E   = S2_PRED.parent / "stage2_end_to_end.tsv"
S2_RUNMANIF = S2_PRED.parent / "stage2_run_manifest.json"
SYN_PATH = Path(_env("C_SYN", "output_synthesis/synthetic_addresses.parquet"))
MISS     = Path(_env("C_MISS", "output/misspellings.tsv"))
DB_PATH  = Path(_env("C_DB", "gnaf.duckdb"))
EXPECTED_TEST_SHA1 = "d48fdfe227fd6ef7828384f020e3042d18873101"

SMOKE       = _env("C_SMOKE", "0") == "1"
ALLOW_CACHE = _env("C_ALLOW_CACHE", "0") == "1"
OUT = Path(_env("C_OUT", "scratch/canine_smoke" if SMOKE else "output_canine"))
PROTECTED = ["output", "output_stage1", "output_stage2", "output_synthesis", "harvest", "cache",
             "archive", "results_labels_746_model_85k", "results_labels_789_model_85k",
             "results_labels_789_model_refit", "output_baseline", "output_baseline_blocked"]
_o = OUT.resolve()
for _p in PROTECTED:
    _r = Path(_p).resolve()
    assert _o != _r and _r not in _o.parents, f"OUT={OUT} is inside protected {_p}; refusing"
OUT.mkdir(parents=True, exist_ok=True)

CANINE_MODEL = _env("C_MODEL", "google/canine-c")
SEED      = 4640
MAX_CHARS = 144            # in characters; the longest training pair is 135 (cell 4)
MINILM_MAX_LEN = 64        # the MiniLM re-ranker's cut-off, in word pieces (cell 8)
EPOCHS    = 2
BATCH     = 16
LR        = 2e-5
FP16      = False
POS_WEIGHT = None
PREDICT_BATCH = 128
VAL_FRAC  = 0.10
MINE_N, MINE_K, NEG_PER_POS = 8_000, 32, 8
FILTER_TEST_ANCHORS = True
RETRIEVE_K = 500
K_MAX  = 10 if SMOKE else 100
K_LIST = [k for k in (10, 20, 50, 100) if k <= K_MAX]
PILOT_STEPS = 5 if SMOKE else 50
MAX_PROJECTED_MIN = 90     # minutes; train only if cell 5 projects less
SMOKE_STEPS = 30
N_SINGLE = 5
FOLLOW = "25 Hoskins Avenue, Kemblawarra NSW 2505"

for label, p in [("Stage 1 model", S1_MODEL), ("Stage 1 per-query", S1_PERQ), ("Stage 1 by-subset", S1_SUB),
                 ("mined pids", MP), ("mined sims", MS), ("mining sidecar", MSIDE),
                 ("cached retrieval", EP), ("retrieval sidecar", ESIDE), ("Stage 2 manifest", S2_MANIF),
                 ("MiniLM model", S2_MODEL), ("MiniLM predictions", S2_PRED), ("MiniLM end-to-end", S2_E2E),
                 ("synthetic set", SYN_PATH), ("test set", MISS), ("database", DB_PATH)]:
    assert p.exists(), f"{label} not found at {p}"
print(f"OUT       : {OUT}   smoke={SMOKE}   allow_cache={ALLOW_CACHE}")
print(f"backbone  : {CANINE_MODEL}, {MAX_CHARS} characters; lr {LR}, {EPOCHS} epochs, batch {BATCH}, seed {SEED}")
print(f"K list    : {K_LIST}   pilot {PILOT_STEPS} steps, ceiling {MAX_PROJECTED_MIN} min")

OUT       : output_canine   smoke=False   allow_cache=True
backbone  : google/canine-c, 144 characters; lr 2e-05, 2 epochs, batch 16, seed 4640
K list    : [10, 20, 50, 100]   pilot 50 steps, ceiling 90 min


In [ ]:
# Cell 2: open the G-NAF database read-only and define RENDER, the query
# normaliser and resolve(), all as in 4_train_reranker.ipynb (the RENDER sha1
# is asserted). S1_MTIME is the time the retriever's weights were saved. The
# cached arrays read below are refused if their record of it differs.

import duckdb, torch
con = duckdb.connect(str(DB_PATH), read_only=True)
RENDER = r"""regexp_replace(trim(concat_ws(' ',
    CASE WHEN g.number_last IS NOT NULL AND g.number_last <> ''
              AND regexp_matches(k.k, '^[0-9]')
         THEN k.k || '-' || g.number_last ELSE k.k END,
    k.street_n, k.type_n, k.suffix_n, k.locality_n, k.state_n, k.postcode)),
    '\s+', ' ', 'g')"""
RJOIN = "LEFT JOIN gnaf_range g ON g.pid = k.pid AND g.number_first = k.k"
RENDER_SHA1 = hashlib.sha1(RENDER.encode()).hexdigest()
assert RENDER_SHA1 == "cc573253b3ead7dddebc8393697909d49f27eaa1", "RENDER differs from the protected one"

def norm(s):
    return re.sub(r"\s+", " ", str(s).upper().replace(",", " ")).strip()

def resolve(pids):
    con.register("need_df", pd.DataFrame({"pid": sorted(set(pids))}))
    return con.sql(f"""
        SELECT n.pid,
               coalesce(min(CASE WHEN k.k NOT LIKE 'LOT %' THEN {RENDER} END), min({RENDER})) AS s,
               min(coalesce(nullif(CAST(k.PRINCIPAL_PID AS VARCHAR), ''), n.pid))              AS principal
        FROM need_df n JOIN gnaf_key k ON k.pid = n.pid {RJOIN}
        GROUP BY n.pid""").df()

device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
_w = sorted(S1_MODEL.rglob("*.safetensors")) + sorted(S1_MODEL.rglob("*.bin"))
S1_MTIME = max(p.stat().st_mtime for p in _w) if _w else 0.0
print(f"database open read-only; RENDER sha1 {RENDER_SHA1[:8]} ok; device {device}; S1_MTIME {S1_MTIME}")

database open read-only; RENDER sha1 cc573253 ok; device mps; S1_MTIME 1788792286.0973577


In [ ]:
# Cell 3: rebuild the training pairs as 4_train_reranker.ipynb cells 5 and 8
# do, from the cached mined candidates. Each mutated address is paired with its
# record (label 1) and with the NEG_PER_POS highest-ranked retriever candidates
# that render differently (label 0). The split is by query string. The counts
# are asserted against the MiniLM re-ranker's manifest.

syn = pd.read_parquet(SYN_PATH)
syn = syn.dropna(subset=["dirty", "anchor"]).drop_duplicates("dirty")
syn["y"] = (~syn.label.str.startswith("NEG")).astype(int)
pos_all = syn[syn.y == 1]
take = min(MINE_N, len(pos_all))
mine = pos_all.sample(take, random_state=SEED)[["dirty", "anchor"]].reset_index(drop=True)
neg_all = syn[syn.y == 0]
frac = take / max(len(pos_all), 1)
syn_neg = neg_all.sample(max(1, int(len(neg_all) * frac)), random_state=SEED)[["dirty", "anchor"]].reset_index(drop=True)
print(f"{len(syn):,} synthetic rows | mining pool {len(mine):,} positives | synthesiser negatives {len(syn_neg):,}")

perq = pd.read_csv(S1_PERQ, sep="\t", dtype={"truth_pid": str})
test_q = perq[["address", "truth_pid", "rank_principal", "top_sim"]].copy()
test_q["q"] = test_q.address.map(norm)
con.register("truth_df", pd.DataFrame({"pid": test_q.truth_pid.unique()}))
_ta = con.sql(f"""SELECT t.pid, {RENDER} AS s FROM truth_df t JOIN gnaf_key k ON k.pid = t.pid {RJOIN}
                  GROUP BY t.pid, s""").df()
TEST_ANCHORS = set(_ta.s) if FILTER_TEST_ANCHORS else set()
print(f"test set {len(test_q):,} matchable queries; test-answer guard {len(TEST_ANCHORS):,} strings")

sc = json.loads(MSIDE.read_text())
assert abs(sc["s1_mtime"] - S1_MTIME) < 1.0, "the mined arrays were produced by a DIFFERENT Stage 1 model; refusing"
assert sc["mine_k"] == MINE_K and sc["n_total"] == len(mine) and sc["n_done"] == sc["n_total"], "mining sidecar disagrees with the quotas"
mined_pids, mined_sims = np.load(MP), np.load(MS)
assert mined_pids.shape == (len(mine), MINE_K)
print(f"mined arrays: {mined_pids.shape[0]:,} queries x {mined_pids.shape[1]} candidates (cached)")

need = sorted({b.decode() if isinstance(b, bytes) else str(b) for row in mined_pids for b in row})
con.register("need_df", pd.DataFrame({"pid": need}))
_r = con.sql(f"""SELECT n.pid, min({RENDER}) AS s FROM need_df n JOIN gnaf_key k ON k.pid = n.pid {RJOIN}
                 GROUP BY n.pid""").df()
REND = dict(zip(_r.pid, _r.s))
from collections import defaultdict
rows, stat = [], defaultdict(int)
for i, r in enumerate(mine.itertuples()):
    rows.append((r.dirty, r.anchor, 1))
    seen, kept = {r.anchor}, 0
    for pid_b, sim in zip(mined_pids[i], mined_sims[i]):
        if kept >= NEG_PER_POS:
            break
        s = REND.get(pid_b.decode() if isinstance(pid_b, bytes) else str(pid_b))
        if s is None:          stat["unrendered"] += 1;   continue
        if s in seen:          stat["same_as_truth_or_dup"] += 1; continue
        if s in TEST_ANCHORS:  stat["test_answer_skipped"] += 1;  continue
        seen.add(s); kept += 1
        rows.append((r.dirty, s, 0))
    stat["negatives_kept"] += kept
for r in syn_neg.itertuples():
    rows.append((r.dirty, r.anchor, 0)); stat["synthesiser_negatives"] += 1
pairs = (pd.DataFrame(rows, columns=["query", "candidate", "label"])
           .drop_duplicates(["query", "candidate"]).reset_index(drop=True))
pairs["grp"] = pairs["query"].map(lambda s: zlib.crc32(s.encode()) % 100)
train_df = pairs[pairs.grp >= VAL_FRAC * 100].drop(columns="grp").reset_index(drop=True)
val_df   = pairs[pairs.grp <  VAL_FRAC * 100].drop(columns="grp").reset_index(drop=True)
assert not (set(train_df["query"]) & set(val_df["query"])), "query leaked across the split"
PAIRS_SHA1 = hashlib.sha1("\n".join(sorted(f"{q}\t{c}\t{l}" for q, c, l in pairs[["query", "candidate", "label"]].itertuples(index=False))).encode()).hexdigest()
_m = json.loads(S2_MANIF.read_text())
print(f"pairs {len(pairs):,}: {int((pairs.label==1).sum()):,} positive, {int((pairs.label==0).sum()):,} negative; "
      f"train {len(train_df):,} / val {len(val_df):,}; skipped test answers {stat['test_answer_skipped']:,}; sha1 {PAIRS_SHA1[:12]}")
assert len(train_df) == _m["n_train_pairs"] and len(val_df) == _m["n_val_pairs"], \
    f"rebuilt {len(train_df)}/{len(val_df)} pairs but Stage 2's manifest says {_m['n_train_pairs']}/{_m['n_val_pairs']}"
print("counts equal Stage 2's manifest: the pairs are the ones MiniLM trained on")
if SMOKE:
    train_df = train_df.head(SMOKE_STEPS * BATCH).reset_index(drop=True); val_df = val_df.head(512).reset_index(drop=True)
    print(f"[smoke] train {len(train_df)} / val {len(val_df)}")

100,000 synthetic rows | mining pool 8,000 positives | synthesiser negatives 1,414


test set 789 matchable queries; test-answer guard 1,139 strings
mined arrays: 8,000 queries x 32 candidates (cached)


pairs 61,029: 8,000 positive, 53,029 negative; train 54,679 / val 6,350; skipped test answers 51; sha1 0c5b72012ce8
counts equal Stage 2's manifest: the pairs are the ones MiniLM trained on


In [ ]:
# Cell 4: check that MAX_CHARS cuts no training pair. CANINE reads
# "[CLS] query [SEP] candidate [SEP]" as characters, so a pair needs
# len(query) + len(candidate) + 3 positions.

_len = (pairs["query"].str.len() + pairs["candidate"].str.len() + 3)
PAIR_CHARS = dict(max=int(_len.max()), p50=int(_len.median()), p99=int(_len.quantile(0.99)), p999=int(_len.quantile(0.999)))
print(f"training pair length in characters: median {PAIR_CHARS['p50']}, p99 {PAIR_CHARS['p99']}, p99.9 {PAIR_CHARS['p999']}, max {PAIR_CHARS['max']}")
assert PAIR_CHARS["max"] <= MAX_CHARS, f"a training pair needs {PAIR_CHARS['max']} characters; raise MAX_CHARS (cell 1) and re-run"
print(f"MAX_CHARS = {MAX_CHARS} cuts nothing")

training pair length in characters: median 80, p99 106, p99.9 117, max 135
MAX_CHARS = 144 cuts nothing


In [ ]:
# Cell 5: time PILOT_STEPS training steps on the real pairs and project the
# full training time, as 4_train_reranker.ipynb cell 9 does. If the projection
# exceeds MAX_PROJECTED_MIN the cell stops and cell 6 does not train. The pilot
# is skipped when C_ALLOW_CACHE=1 and OUT/model exists.

import transformers
from datasets import Dataset
from sentence_transformers.cross_encoder import (CrossEncoder, CrossEncoderTrainer, CrossEncoderTrainingArguments)
from sentence_transformers.cross_encoder.losses import BinaryCrossEntropyLoss
_tf5  = int(transformers.__version__.split(".")[0]) >= 5
_warm = dict(warmup_steps=0.1) if _tf5 else dict(warmup_ratio=0.1)

def to_ds(df):
    return Dataset.from_pandas(df.assign(label=df.label.astype("float32"))[["query", "candidate", "label"]],
                               preserve_index=False)

PROJECTED_MIN, SEC_PER_STEP = float("nan"), float("nan")
if ALLOW_CACHE and (OUT / "model").exists():
    print("[cache] C_ALLOW_CACHE=1 and OUT/model exists: pilot skipped")
else:
    _probe = CrossEncoder(CANINE_MODEL, num_labels=1, max_length=MAX_CHARS, device=device)
    _args = CrossEncoderTrainingArguments(
        output_dir=str(OUT / "_probe"), max_steps=PILOT_STEPS,
        per_device_train_batch_size=BATCH, learning_rate=LR,
        fp16=FP16, bf16=False, eval_strategy="no", save_strategy="no",
        logging_steps=PILOT_STEPS, seed=SEED, dataloader_num_workers=0, report_to="none")
    _t0 = time.time()
    CrossEncoderTrainer(model=_probe, args=_args, train_dataset=to_ds(train_df.head(PILOT_STEPS * BATCH)),
                        loss=BinaryCrossEntropyLoss(_probe)).train()
    SEC_PER_STEP = (time.time() - _t0) / PILOT_STEPS
    steps = int(np.ceil(len(train_df) / BATCH)) * EPOCHS if not SMOKE else SMOKE_STEPS
    PROJECTED_MIN = steps * SEC_PER_STEP / 60
    print(f"\nmeasured  : {SEC_PER_STEP:.2f} s per step at batch {BATCH}, {MAX_CHARS} characters")
    print(f"full run  : {steps:,} steps over {EPOCHS} epochs")
    print(f"PROJECTED : {PROJECTED_MIN:.0f} min ({PROJECTED_MIN/60:.2f} hours); ceiling {MAX_PROJECTED_MIN} min")
    del _probe
    if not SMOKE and PROJECTED_MIN > MAX_PROJECTED_MIN:
        raise SystemExit(f"projected {PROJECTED_MIN:.0f} min exceeds the {MAX_PROJECTED_MIN} min limit; not training")

[cache] C_ALLOW_CACHE=1 and OUT/model exists: pilot skipped


In [ ]:
# Cell 6: fine-tune the CANINE re-ranker with the training arguments of
# 4_train_reranker.ipynb cell 10. A one-unit head learns the 0/1 label under
# binary cross-entropy. Saves OUT/model and OUT/canine_manifest.json. Takes
# about an hour. With C_ALLOW_CACHE=1 the saved model is loaded instead.

TRAIN_MIN = float("nan")
if ALLOW_CACHE and (OUT / "model").exists():
    ce_model = CrossEncoder(str(OUT / "model"), num_labels=1, max_length=MAX_CHARS, device=device)
    print(f"[cache] loaded the CANINE cross-encoder from {OUT/'model'}; canine_manifest.json left untouched")
else:
    assert SMOKE or PROJECTED_MIN <= MAX_PROJECTED_MIN, "cell 5 refused; not training"
    ce_model = CrossEncoder(CANINE_MODEL, num_labels=1, max_length=MAX_CHARS, device=device)
    loss = (BinaryCrossEntropyLoss(ce_model, pos_weight=torch.tensor(float(POS_WEIGHT))) if POS_WEIGHT
            else BinaryCrossEntropyLoss(ce_model))
    _extra = dict(max_steps=SMOKE_STEPS) if SMOKE else dict(num_train_epochs=EPOCHS)
    args = CrossEncoderTrainingArguments(
        output_dir=str(OUT / "checkpoints"), **_extra,
        per_device_train_batch_size=BATCH, per_device_eval_batch_size=BATCH * 4,
        learning_rate=LR, **_warm, fp16=FP16, bf16=False,
        eval_strategy="epoch" if not SMOKE else "no",
        save_strategy="epoch" if not SMOKE else "no", save_total_limit=2,
        logging_steps=50, seed=SEED, dataloader_num_workers=0, report_to="none")
    trainer = CrossEncoderTrainer(model=ce_model, args=args, train_dataset=to_ds(train_df),
                                  eval_dataset=to_ds(val_df), loss=loss)
    _t0 = time.time()
    trainer.train()
    TRAIN_MIN = (time.time() - _t0) / 60
    ce_model.save_pretrained(str(OUT / "model"))
    (OUT / "canine_manifest.json").write_text(json.dumps(dict(
        base_model=CANINE_MODEL, epochs=EPOCHS, batch=BATCH, lr=LR, max_chars=MAX_CHARS,
        pos_weight=POS_WEIGHT, n_train_pairs=len(train_df), n_val_pairs=len(val_df), pairs_sha1=PAIRS_SHA1,
        mine_n=len(mine), mine_k=MINE_K, neg_per_pos=NEG_PER_POS, filter_test_anchors=FILTER_TEST_ANCHORS,
        s1_mtime=S1_MTIME, seed=SEED, smoke=SMOKE, sec_per_step_pilot=round(SEC_PER_STEP, 3),
        projected_minutes=round(PROJECTED_MIN, 1), train_minutes=round(TRAIN_MIN, 1),
        timestamp=time.strftime("%Y-%m-%d %H:%M:%S")), indent=2))
    print(f"\ntrained in {TRAIN_MIN:.0f} min, saved to {OUT/'model'}")

[cache] loaded the CANINE cross-encoder from output_canine/model; canine_manifest.json left untouched


In [ ]:
# Cell 7: assemble the 950 test-set queries as 4_train_reranker.ipynb cell 11
# does (789 matchable, 129 no_number, 32 no_record). Load the cached retriever
# candidates (950 x 500) and refuse them if their record of the model, query
# count, K or query list differs. This notebook does not retrieve. Then render
# the top K_MAX candidates of each query, as 4_train_reranker.ipynb cell 13 does.

TEST_SHA1 = hashlib.sha1(MISS.read_bytes()).hexdigest()
if MISS == Path("output/misspellings.tsv"):
    assert TEST_SHA1 == EXPECTED_TEST_SHA1, f"test set sha1 {TEST_SHA1} is not the expected test set"
full = pd.read_csv(MISS, sep="\t", dtype=str).fillna("")
assert full.address.is_unique
_exc = full.adjudication.str.contains("excluded", regex=False) if "adjudication" in full.columns else pd.Series(False, index=full.index)
no_rec = full.reason.str.contains("Cannot be matched") & ~_exc
no_num = full.reason.str.contains("Street number does not exist") & ~_exc
unmatch = full[no_rec | no_num].copy()
unmatch["kind"] = np.where(unmatch.reason.str.contains("Cannot be matched"), "no_record", "no_number")
unmatch["population"] = "unmatchable"; unmatch["truth_pid"] = ""
excl = full[_exc].copy(); excl["kind"] = "excluded"; excl["population"] = "excluded"; excl["truth_pid"] = ""
match = test_q.copy(); match["kind"] = "matchable"; match["population"] = "matchable"; match["reason"] = ""
allq = pd.concat([match[["address", "q", "truth_pid", "population", "kind", "reason"]],
                  unmatch.assign(q=unmatch.address.map(norm))[["address", "q", "truth_pid", "population", "kind", "reason"]],
                  excl.assign(q=excl.address.map(norm))[["address", "q", "truth_pid", "population", "kind", "reason"]]],
                 ignore_index=True)
allq["uid"] = allq.index
assert len(allq) == len(full) and set(allq.address) == set(full.address)
m_uids = allq.loc[allq.population == "matchable", "uid"].to_numpy()
u_uids = allq.loc[allq.population == "unmatchable", "uid"].to_numpy()
uid2addr = dict(zip(allq.uid, allq.address))
FOLLOW_UID = int(allq.loc[allq.address == FOLLOW, "uid"].iloc[0])
for k_, g in allq.groupby("kind"): print(f"  {k_:12s} {len(g):5d}")

Q_HASH = hashlib.sha1("\n".join(allq.q.tolist()).encode()).hexdigest()
sc = json.loads(ESIDE.read_text())
_checks = {"model mtime": abs(sc.get("s1_mtime", -1) - S1_MTIME) < 1.0, "n queries": sc.get("n") == len(allq),
           "K": sc.get("k") == RETRIEVE_K, "query list": sc.get("q_hash") == Q_HASH}
assert all(_checks.values()), f"cached retrieval is stale ({[k for k, ok in _checks.items() if not ok]}); 4_train_reranker.ipynb cell 12 must re-run it; this notebook never retrieves"
ev_pids, ev_sims = np.load(EP), np.load(ES)
assert ev_pids.shape == (len(allq), RETRIEVE_K)
print(f"cached Stage 1 retrieval reused: {ev_pids.shape[0]:,} queries x {ev_pids.shape[1]} (all four sidecar checks pass)")

need = sorted({b.decode() if isinstance(b, bytes) else str(b) for row in ev_pids[:, :K_MAX] for b in row} | set(match.truth_pid))
_res = resolve(need)
REND2 = dict(zip(_res.pid, _res.s)); P2P = dict(zip(_res.pid, _res.principal))
_dec = np.vectorize(lambda b: b.decode() if isinstance(b, bytes) else str(b))
raw_rows = []
for i in range(len(allq)):
    pl = _dec(ev_pids[i]); sl = ev_sims[i]
    for r in range(min(K_MAX, len(pl))):
        raw_rows.append((i, r + 1, pl[r], REND2.get(pl[r], ""), float(sl[r])))
cand_raw = pd.DataFrame(raw_rows, columns=["uid", "rank", "pid", "cand", "sim"])
TRUTH_PID = dict(zip(match.address, match.truth_pid))
truth_pr  = {u: P2P.get(TRUTH_PID[uid2addr[u]], TRUTH_PID[uid2addr[u]]) for u in m_uids}
truth_str = {u: REND2.get(TRUTH_PID[uid2addr[u]], "") for u in m_uids}
_tl = (cand_raw.uid.map(dict(zip(allq.uid, allq.q))).str.len() + cand_raw.cand.str.len() + 3)
print(f"raw candidates: {len(cand_raw):,} pairs ({len(cand_raw)/len(allq):.0f} per query); longest test pair {int(_tl.max())} characters "
      f"({'fits' if _tl.max() <= MAX_CHARS else 'CUT'} in {MAX_CHARS})")
assert _tl.max() <= MAX_CHARS
print(f"followed address uid {FOLLOW_UID}; truth {truth_str[FOLLOW_UID]!r}")

  matchable      789
  no_number      129
  no_record       32
cached Stage 1 retrieval reused: 950 queries x 500 (all four sidecar checks pass)


raw candidates: 95,000 pairs (100 per query); longest test pair 125 characters (fits in 144)
followed address uid 773; truth '25 HOSKINS AVENUE WARRAWONG NSW 2502'


In [ ]:
# Cell 8: score every candidate pair with the CANINE re-ranker, the MiniLM
# re-ranker (at its own 64-piece cut-off) and RapidFuzz WRatio, so all three
# see identical inputs. Then time both re-rankers on the 9,500 K=10 pairs as
# one batch and on one query's 10 pairs alone (N_SINGLE queries). Scoring all
# 95,000 pairs with CANINE took 26.7 minutes in the saved run.

from rapidfuzz import fuzz
QMAP = dict(zip(allq.uid, allq.q))
qs = cand_raw.uid.map(QMAP).tolist(); cs = cand_raw.cand.tolist()

# canine_scores() pads every pair to MAX_CHARS, because CANINE leaves the end
# of an unpadded pair out of its main layers. The saved model applies a sigmoid of its own, so
# the raw logits are read from the inner model and one sigmoid is applied here.
# The MiniLM re-ranker's saved activation is Identity, so predict() is used.
_hf = next(m for m in ce_model.modules() if m.__class__.__name__ == "CanineForSequenceClassification")
_hf.eval()
@torch.no_grad()
def canine_scores(pairs, bs=PREDICT_BATCH):
    out = []
    for i in range(0, len(pairs), bs):
        a, b = zip(*pairs[i:i + bs])
        enc = ce_model.tokenizer(list(a), list(b), padding="max_length", truncation=True, max_length=MAX_CHARS, return_tensors="pt")
        enc = {k: v.to(_hf.device) for k, v in enc.items()}
        out.append(_hf(**enc).logits.squeeze(-1).float().cpu().numpy())
    lg = np.concatenate(out).astype(np.float64)
    return 1.0 / (1.0 + np.exp(-lg))

t0 = time.perf_counter()
cand_raw["s_canine"] = canine_scores(list(zip(qs, cs)))
assert cand_raw.s_canine.between(0, 1).all() and cand_raw.s_canine.min() < 0.45, "CANINE scores do not span the 0-to-1 scale; is a sigmoid applied twice?"
T_CANINE_ALL = time.perf_counter() - t0
minilm = CrossEncoder(str(S2_MODEL), num_labels=1, max_length=MINILM_MAX_LEN, device=device)
t0 = time.perf_counter()
_lg = minilm.predict(list(zip(qs, cs)), batch_size=PREDICT_BATCH, show_progress_bar=False)
cand_raw["s_minilm"] = 1.0 / (1.0 + np.exp(-np.asarray(_lg, dtype=np.float64)))
T_MINILM_ALL = time.perf_counter() - t0
cand_raw["s_fuzz"] = [fuzz.WRatio(a, b) / 100.0 for a, b in zip(qs, cs)]
print(f"scored {len(cand_raw):,} pairs: CANINE {T_CANINE_ALL/60:.1f} min ({T_CANINE_ALL/len(cand_raw)*1000:.1f} ms/pair), "
      f"MiniLM {T_MINILM_ALL:.1f} s ({T_MINILM_ALL/len(cand_raw)*1000:.2f} ms/pair)")
print(f"CANINE score distribution: p5 {cand_raw.s_canine.quantile(.05):.3f}  p50 {cand_raw.s_canine.median():.3f}  p95 {cand_raw.s_canine.quantile(.95):.3f}")

# timing: the K=10 pairs as one batch, then one query's pairs alone
sub10 = cand_raw[cand_raw["rank"] <= 10]
p10 = list(zip(sub10.uid.map(QMAP), sub10.cand))
_rng = np.random.default_rng(SEED)
SINGLE_UIDS = [FOLLOW_UID] + sorted(_rng.choice([u for u in m_uids if u != FOLLOW_UID], size=N_SINGLE - 1, replace=False).tolist())
TIMING = []
for name, model in [("canine", ce_model), ("minilm", minilm)]:
    _score = canine_scores if name == "canine" else (lambda pp, model=model: model.predict(pp, batch_size=PREDICT_BATCH, show_progress_bar=False))
    t0 = time.perf_counter(); _score(p10); batch_s = time.perf_counter() - t0
    singles = []
    for u in SINGLE_UIDS:
        pp = [(QMAP[u], c) for c in sub10.loc[sub10.uid == u, "cand"]]
        t0 = time.perf_counter(); _score(pp); singles.append(time.perf_counter() - t0)
    TIMING.append(dict(model=name, n_pairs=len(p10), batch_s=round(batch_s, 3), per_query_ms=round(batch_s / len(allq) * 1000, 2),
                       single_s_mean=round(float(np.mean(singles)), 4), single_s_runs=json.dumps([round(x, 4) for x in singles]),
                       all_pairs=len(cand_raw), all_pairs_s=round(T_CANINE_ALL if name == "canine" else T_MINILM_ALL, 1)))
    print(f"  {name:7s} K=10 batch of {len(p10):,} pairs {batch_s:6.1f}s ({batch_s/len(allq)*1000:5.1f} ms/query) | one query's 10 pairs alone {np.mean(singles)*1000:.0f} ms")
pd.DataFrame(TIMING).to_csv(OUT / "canine_timing.tsv", sep="\t", index=False)
cand_raw.to_csv(OUT / "canine_candidates.tsv", sep="\t", index=False)
print(f"wrote canine_candidates.tsv and canine_timing.tsv to {OUT}")

scored 95,000 pairs: CANINE 26.7 min (16.8 ms/pair), MiniLM 88.1 s (0.93 ms/pair)
CANINE score distribution: p5 0.001  p50 0.001  p95 0.887


  canine  K=10 batch of 9,500 pairs  154.9s (163.1 ms/query) | one query's 10 pairs alone 150 ms


  minilm  K=10 batch of 9,500 pairs    8.8s (  9.2 ms/query) | one query's 10 pairs alone 36 ms


wrote canine_candidates.tsv and canine_timing.tsv to output_canine


In [ ]:
# Cell 9: the end-to-end table, as in 4_train_reranker.ipynb cell 15. For each
# K and each scorer: top-1 under the pid rule and the string rule, MRR, and
# recall@5 and @10, over all matchable queries, with a Wilson interval on the
# string-rule top-1. Ties keep the retriever's order. The MiniLM rows are
# checked against the MiniLM re-ranker's saved stage2_end_to_end.tsv.

def wilson(p, n, z=1.96):
    d = 1 + z*z/n; c = p + z*z/(2*n); h = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n))
    return (c - h)/d, (c + h)/d

def evaluate(df, K, col, variant="raw"):
    sub = df[(df["rank"] <= K) & (df.uid.isin(m_uids))]
    ordered = sub.sort_values(["uid", col, "rank"], ascending=[True, False, True], kind="mergesort")
    top = ordered.groupby("uid").head(1).set_index("uid")
    hit_pid = hit_str = 0; rr, r5, r10 = [], 0, 0
    for u in m_uids:
        if u not in top.index:
            rr.append(0.0); continue
        row = top.loc[u]
        if P2P.get(row.pid, row.pid) == truth_pr[u]: hit_pid += 1
        if row.cand == truth_str[u]:                hit_str += 1
        g = ordered[ordered.uid == u]
        pos = np.flatnonzero(g.cand.to_numpy() == truth_str[u])
        if len(pos):
            rr.append(1.0 / (pos[0] + 1)); r5 += pos[0] < 5; r10 += pos[0] < 10
        else:
            rr.append(0.0)
    n = len(m_uids); lo, hi = wilson(hit_str / n, n)
    return dict(variant=variant, scorer=col.replace("s_", ""), K=K, top1_pid=hit_pid / n, top1_str=hit_str / n,
                hit_str=hit_str, hit_pid=hit_pid, ci_lo=lo, ci_hi=hi, mrr=float(np.mean(rr)), r_at_5=r5 / n, r_at_10=r10 / n)

e2e = pd.DataFrame([evaluate(cand_raw, K, col) for K in K_LIST for col in ["s_canine", "s_minilm", "s_fuzz"]])
print(f"END-TO-END, all {len(m_uids)} matchable queries in the denominator (raw candidates)")
print(e2e.round(4).to_string(index=False))
e2e.to_csv(OUT / "canine_end_to_end.tsv", sep="\t", index=False)
best_c = e2e[e2e.scorer == "canine"].nlargest(1, "top1_str").iloc[0]
K_REPORT_C = int(best_c.K)
print(f"\n*** CANINE best K={K_REPORT_C}: top-1 string {best_c.top1_str:.4f} [{best_c.ci_lo:.3f}, {best_c.ci_hi:.3f}], pid {best_c.top1_pid:.4f} ***")
_arch = pd.read_csv(S2_E2E, sep="\t")
_arch = _arch[(_arch.variant == "raw") & (_arch.scorer == "model")].set_index("K")
_here = e2e[e2e.scorer == "minilm"].set_index("K")
_d = max(abs(_here.top1_str[k] - _arch.top1_str[k]) for k in K_LIST if k in _arch.index)
print(f"MiniLM re-scored here against the archived table: max |top-1 string difference| {_d:.4f} over K {K_LIST}")

END-TO-END, all 789 matchable queries in the denominator (raw candidates)
variant scorer   K  top1_pid  top1_str  hit_str  hit_pid  ci_lo  ci_hi    mrr  r_at_5  r_at_10
    raw canine  10    0.5361    0.8327      657      423 0.8051 0.8571 0.8592  0.8872   0.9075
    raw minilm  10    0.5323    0.8276      653      420 0.7997 0.8524 0.8562  0.8872   0.9075
    raw   fuzz  10    0.4728    0.7490      591      373 0.7176 0.7780 0.7980  0.8657   0.9075
    raw canine  20    0.5311    0.8200      647      419 0.7917 0.8453 0.8542  0.8948   0.9062
    raw minilm  20    0.5285    0.8188      646      417 0.7904 0.8441 0.8519  0.8897   0.9049
    raw   fuzz  20    0.4588    0.7262      573      362 0.6941 0.7562 0.7790  0.8327   0.8796
    raw canine  50    0.5158    0.8010      632      407 0.7717 0.8274 0.8378  0.8809   0.8961
    raw minilm  50    0.5222    0.8137      642      412 0.7850 0.8393 0.8485  0.8910   0.9024
    raw   fuzz  50    0.4360    0.6717      530      344 0.6382 0.7036 

In [ ]:
# Cell 10: fit tau for the CANINE re-ranker as 4_train_reranker.ipynb cell 17
# does, by maximum F1 on the synthetic validation pairs, never on the test set.
# Then the confusion counts at tau at K_REPORT_C and at K=10, AUROC and a
# diagnostic sweep. Last, three thresholds for both re-rankers at K=10, as in
# 5_baselines.ipynb: the best test-set threshold, five-fold cross-validation
# and the re-ranker's own tau.

sys.path.insert(0, "tools")
from baseline_metrics import confusion, best_threshold, cv_accuracy, auroc

# the validation scores are saved so the tau fit can be checked
vs = canine_scores(list(zip(val_df["query"], val_df.candidate))); yv = val_df.label.to_numpy()
pd.DataFrame({"score": vs, "label": yv}).to_csv(OUT / "canine_val_scores.tsv", sep="\t", index=False)
def f1_at(t):
    p = (vs >= t).astype(int)
    tp = int(((p == 1) & (yv == 1)).sum()); fp = int(((p == 1) & (yv == 0)).sum()); fn = int(((p == 0) & (yv == 1)).sum())
    return 2 * tp / max(2 * tp + fp + fn, 1)
grid = np.arange(0.05, 1.00, 0.01)
TAU_C = float(grid[int(np.argmax([f1_at(t) for t in grid]))])
print(f"tau (CANINE) = {TAU_C:.2f}  (max-F1 on {len(val_df):,} SYNTHETIC validation pairs)")
TAU_MINILM = float(json.loads(S2_RUNMANIF.read_text())["eval"]["tau"])

def top1_at(K, col):
    sub = cand_raw[cand_raw["rank"] <= K]
    top = (sub.sort_values(["uid", col, "rank"], ascending=[True, False, True], kind="mergesort").groupby("uid").head(1).reset_index(drop=True))
    top = top.merge(allq[["uid", "address", "population", "kind", "reason"]], on="uid")
    top = top[top.population != "excluded"].reset_index(drop=True)
    top["is_truth"] = [r.cand == truth_str.get(r.uid, "\x00") for r in top.itertuples()]
    return top

def report(t, col, tau):
    a, m = t[col] >= tau, t.population == "matchable"
    ca = int((a & t.is_truth & m).sum()); wa_w = int((a & m & ~t.is_truth).sum()); wa_u = int((a & ~m).sum())
    rm = int((~a & m).sum()); cr = int((~a & ~m).sum())
    prec = ca / max(ca + wa_w + wa_u, 1); rec = ca / max(int(m.sum()), 1); spec = cr / max(int((~m).sum()), 1)
    return dict(tau=round(tau, 2), coverage=round(float(a.mean()), 4), precision=round(prec, 4), recall=round(rec, 4),
                specificity=round(spec, 4), f1=round(2*prec*rec/max(prec+rec, 1e-9), 4),
                correct_accept=ca, wrong_accept_wrong_record=wa_w, wrong_accept_unmatchable=wa_u,
                rejected_matchable=rm, correct_reject=cr)

top_c = top1_at(K_REPORT_C, "s_canine")
top_c10 = top1_at(10, "s_canine")
mp = top_c10.loc[top_c10.population == "matchable", "s_canine"].to_numpy(); up = top_c10.loc[top_c10.population == "unmatchable", "s_canine"].to_numpy()
print(f"AUROC matchable vs unmatchable (CANINE top score at K=10): {auroc(mp, up):.4f}")
for K_, t_ in ((K_REPORT_C, top_c), (10, top_c10)):
    print(f"\nAT THE FITTED THRESHOLD (tau = {TAU_C:.2f}, K = {K_})")
    for k, v in report(t_, "s_canine", TAU_C).items(): print(f"  {k:26s} {v}")
    if K_ == 10 and K_REPORT_C == 10: break
print("\nDIAGNOSTIC SWEEP at K=10 (not the selection procedure)")
print(pd.DataFrame([report(top_c10, "s_canine", t) for t in [0.1, 0.3, 0.5, 0.7, 0.8, 0.9, 0.95]]).to_string(index=False))
_out = top_c10.rename(columns={"s_canine": "s_model"})[["uid", "rank", "pid", "cand", "sim", "s_model", "s_fuzz", "address", "population", "kind", "reason", "is_truth"]]
_out.to_csv(OUT / "canine_predictions.tsv", sep="\t", index=False)
top_c.rename(columns={"s_canine": "s_model"}).to_csv(OUT / f"canine_predictions_K{K_REPORT_C}.tsv", sep="\t", index=False)

# the three thresholds, for both re-rankers at K=10
arch = pd.read_csv(S2_PRED, sep="\t", dtype={"pid": str}).merge(allq[["address", "uid"]].rename(columns={"uid": "uid_here"}), on="address")
arch = arch[arch.population != "excluded"]
rows_t = []
for name, t, col, tau in (("canine_K10", top_c10, "s_canine", TAU_C), ("minilm_K10_archived", arch, "s_model", TAU_MINILM)):
    m = (t.population == "matchable").to_numpy(); sc = t[col].to_numpy(float); ok = t.is_truth.astype(bool).to_numpy()
    bt, acc = best_threshold(sc, ok, m); c = confusion(sc, ok, m, bt); cv = cv_accuracy(sc, ok, m, t.population.to_numpy(), seed=SEED)
    own = confusion(sc, ok, m, tau)
    rows_t.append(dict(model=name, n=len(t), top1_str=int((ok & m).sum()), auroc=auroc(sc[m], sc[~m]), best_t=bt, best_accurate=acc,
                       cv_accurate=cv["accurate"], cv_t_min=min(cv["thresholds"]), cv_t_max=max(cv["thresholds"]),
                       own_tau=tau, own_tau_accurate=own["accurate"],
                       **{f"own_{k}": v for k, v in own.items() if k in ("correct_accept", "wrong_accept_wrong_record", "wrong_accept_unmatchable", "rejected_matchable", "correct_reject")}))
thr = pd.DataFrame(rows_t); thr.to_csv(OUT / "canine_threshold_rows.tsv", sep="\t", index=False)
print("\nBASELINE-STYLE ROWS (of 950): top-1 | AUROC | best-threshold right | cross-validated right | own-tau right")
print(thr[["model", "top1_str", "auroc", "best_t", "best_accurate", "cv_accurate", "own_tau", "own_tau_accurate"]].round(4).to_string(index=False))

tau (CANINE) = 0.84  (max-F1 on 6,350 SYNTHETIC validation pairs)
AUROC matchable vs unmatchable (CANINE top score at K=10): 0.8931

AT THE FITTED THRESHOLD (tau = 0.84, K = 10)
  tau                        0.84
  coverage                   0.7147
  precision                  0.8616
  recall                     0.7414
  specificity                0.8696
  f1                         0.797
  correct_accept             585
  wrong_accept_wrong_record  73
  wrong_accept_unmatchable   21
  rejected_matchable         131
  correct_reject             140

DIAGNOSTIC SWEEP at K=10 (not the selection procedure)
 tau  coverage  precision  recall  specificity     f1  correct_accept  wrong_accept_wrong_record  wrong_accept_unmatchable  rejected_matchable  correct_reject
0.10    0.7842     0.8322  0.7858       0.7516 0.8083             620                         85                        40                  84             121
0.30    0.7611     0.8409  0.7706       0.7950 0.8042             608   

In [ ]:
# Cell 11: compare the CANINE re-ranker with the retriever alone and with the
# MiniLM re-ranker at K=10, overall and per subset, under the string rule.
# Counts lost and rescued queries and runs McNemar's exact test, with the
# function from 4_train_reranker.ipynb cell 15c.

def mcnemar_exact(b, c):
    m = b + c
    if m == 0: return 1.0
    k = min(b, c)
    return min(1.0, 2 * sum(math.comb(m, i) for i in range(k + 1)) / (2 ** m))

gg = cand_raw[cand_raw.uid.isin(m_uids)].sort_values(["uid", "rank"], kind="mergesort")
s1_str_rank = {}
for uid, grp in gg.groupby("uid", sort=False):
    pos = np.flatnonzero(grp.cand.to_numpy() == truth_str[uid])
    s1_str_rank[uid] = int(grp["rank"].to_numpy()[pos[0]]) if len(pos) else -1
s1_hit = np.array([s1_str_rank[u] == 1 for u in m_uids])
c10 = dict(zip(top_c10.uid, top_c10.is_truth)); cK = dict(zip(top_c.uid, top_c.is_truth))
mini = dict(zip(arch.uid_here, arch.is_truth.astype(bool)))
canine_hit10 = np.array([bool(c10.get(u, False)) for u in m_uids]); canine_hitK = np.array([bool(cK.get(u, False)) for u in m_uids])
minilm_hit = np.array([bool(mini.get(u, False)) for u in m_uids])
sb = pd.read_csv(S1_SUB, sep="\t", dtype=str); a2s = dict(zip(sb.address, sb.subset))
bysub_c = pd.DataFrame(dict(uid=m_uids, address=[uid2addr[u] for u in m_uids], subset=[a2s.get(uid2addr[u], "unknown") for u in m_uids],
                            s1_str_rank=[s1_str_rank[u] for u in m_uids], s1_hit=s1_hit, minilm_hit_K10=minilm_hit,
                            canine_hit_K10=canine_hit10))
if K_REPORT_C != 10:
    bysub_c[f"canine_hit_K{K_REPORT_C}"] = canine_hitK
bysub_c.to_csv(OUT / "canine_by_subset.tsv", sep="\t", index=False)

print(f"STRING RULE, LIKE FOR LIKE, K=10 (n={len(m_uids)})")
print(f"  Stage 1 alone : {int(s1_hit.sum()):4d}  ({s1_hit.mean():.4f})")
print(f"  MiniLM        : {int(minilm_hit.sum()):4d}  ({minilm_hit.mean():.4f})  archived")
print(f"  CANINE        : {int(canine_hit10.sum()):4d}  ({canine_hit10.mean():.4f})  (at its best K={K_REPORT_C}: {int(canine_hitK.sum())})")
recs = []
for name in ["single", "range_first", "range_last", "range_interior", "range_written", "ALL"]:
    msk = np.ones(len(bysub_c), bool) if name == "ALL" else (bysub_c.subset == name).to_numpy()
    if not msk.sum(): continue
    for pair, a_, b_ in (("canine_vs_stage1", s1_hit, canine_hit10), ("canine_vs_minilm", minilm_hit, canine_hit10)):
        lost = int((a_[msk] & ~b_[msk]).sum()); resc = int((~a_[msk] & b_[msk]).sum()); p = mcnemar_exact(lost, resc)
        recs.append(dict(comparison=pair, subset=name, n=int(msk.sum()), before=float(a_[msk].mean()), after=float(b_[msk].mean()),
                         lost=lost, rescued=resc, p=p, verdict="significant" if p < 0.05 else ("underpowered" if lost + resc < 10 else "no effect")))
mcn = pd.DataFrame(recs); mcn.to_csv(OUT / "canine_mcnemar.tsv", sep="\t", index=False)
for pair, title in (("canine_vs_stage1", "CANINE against STAGE 1 ALONE"), ("canine_vs_minilm", "CANINE against MINILM (both at K=10)")):
    print(f"\n{title}: lost = the first had it right and CANINE did not; rescued = the reverse")
    print(mcn[mcn.comparison == pair].drop(columns="comparison").round(4).to_string(index=False))

STRING RULE, LIKE FOR LIKE, K=10 (n=789)
  Stage 1 alone :  641  (0.8124)
  MiniLM        :  653  (0.8276)  archived
  CANINE        :  657  (0.8327)  (at its best K=10: 657)

CANINE against STAGE 1 ALONE: lost = the first had it right and CANINE did not; rescued = the reverse
        subset   n  before  after  lost  rescued      p      verdict
        single 634  0.8502 0.8533    27       29 0.8939    no effect
   range_first  33  0.4242 0.6667     2       10 0.0386  significant
    range_last  54  0.7593 0.8889     2        9 0.0654    no effect
range_interior  20  0.2500 0.2500     1        1 1.0000 underpowered
 range_written  45  0.8667 0.8444     3        2 1.0000 underpowered
           ALL 789  0.8124 0.8327    35       51 0.1052    no effect

CANINE against MINILM (both at K=10): lost = the first had it right and CANINE did not; rescued = the reverse
        subset   n  before  after  lost  rescued      p      verdict
        single 634  0.8312 0.8533    17       31 0.0595    

In [ ]:
# Cell 12: one address through both re-rankers. Prints the followed address's
# 10 candidates with the retriever similarity and both re-rankers' scores, then
# each re-ranker's pick and whether it is accepted at that re-ranker's tau.

u = FOLLOW_UID
g = cand_raw[(cand_raw.uid == u) & (cand_raw["rank"] <= 10)].sort_values("rank")
print(f"query : {QMAP[u]!r}\ntruth : {truth_str[u]!r}\n")
print(f"{'rank':>4s} {'S1 sim':>7s} {'MiniLM':>7s} {'CANINE':>7s}  candidate")
for r in g.itertuples():
    mark = "TRUE" if r.cand == truth_str[u] else "    "
    print(f"{r.rank:4d} {r.sim:7.3f} {r.s_minilm:7.3f} {r.s_canine:7.3f}  {mark} {r.cand}")
for name, col, tau in (("MiniLM", "s_minilm", TAU_MINILM), ("CANINE", "s_canine", TAU_C)):
    pick = g.sort_values([col, "rank"], ascending=[False, True]).iloc[0]
    print(f"{name}: picks rank {int(pick['rank'])} ({'TRUE' if pick.cand == truth_str[u] else 'wrong'}) at {pick[col]:.3f}, "
          f"{'accepted' if pick[col] >= tau else 'refused'} at its tau {tau:.2f}")

query : '25 HOSKINS AVENUE KEMBLAWARRA NSW 2505'
truth : '25 HOSKINS AVENUE WARRAWONG NSW 2502'

rank  S1 sim  MiniLM  CANINE  candidate
   1   0.777   0.917   0.079  TRUE 25 HOSKINS AVENUE WARRAWONG NSW 2502
   2   0.708   0.068   0.002       25 HOSKINS STREET NYNGAN NSW 2825
   3   0.689   0.132   0.015       25 HOSKINS AVENUE LITHGOW NSW 2790
   4   0.689   0.132   0.015       25 HOSKINS AVENUE LITHGOW NSW 2790
   5   0.689   0.132   0.015       25 HOSKINS AVENUE LITHGOW NSW 2790
   6   0.689   0.132   0.015       25 HOSKINS AVENUE LITHGOW NSW 2790
   7   0.689   0.132   0.015       25 HOSKINS AVENUE LITHGOW NSW 2790
   8   0.689   0.132   0.015       25 HOSKINS AVENUE LITHGOW NSW 2790
   9   0.689   0.132   0.015       25 HOSKINS AVENUE LITHGOW NSW 2790
  10   0.689   0.132   0.015       25 HOSKINS AVENUE LITHGOW NSW 2790
MiniLM: picks rank 1 (TRUE) at 0.917, accepted at its tau 0.83
CANINE: picks rank 1 (TRUE) at 0.079, refused at its tau 0.84


In [ ]:
# Cell 13: write OUT/canine_run_manifest.json, a record of what this run read
# and produced, and close the database.

import subprocess, platform, transformers, sentence_transformers
try:
    _git = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
except Exception:
    _git = ""
_cm = json.loads((OUT / "canine_manifest.json").read_text()) if (OUT / "canine_manifest.json").exists() else {}
manifest = dict(
    timestamp=time.strftime("%Y-%m-%d %H:%M:%S"), smoke=SMOKE, git_hash=_git,
    backbone=CANINE_MODEL, max_chars=MAX_CHARS, epochs=EPOCHS, batch=BATCH, lr=LR, seed=SEED,
    training=_cm, pairs_sha1=PAIRS_SHA1, pair_chars=(PAIR_CHARS if 'PAIR_CHARS' in dir() else {}),
    test_file=str(MISS), test_sha1=TEST_SHA1, n_rows=int(len(allq)), n_matchable=int(len(m_uids)), n_unmatchable=int(len(u_uids)),
    retrieval=dict(file=str(EP), sidecar=json.loads(ESIDE.read_text())), s1_mtime=S1_MTIME,
    k_max=K_MAX, k_list=K_LIST, k_report_canine=K_REPORT_C, tau_canine=TAU_C, tau_minilm_archived=TAU_MINILM,
    minilm_predictions=str(S2_PRED), single_uids=[int(x) for x in SINGLE_UIDS],
    canine_top1_str_K10=int(canine_hit10.sum()), canine_top1_str_bestK=int(canine_hitK.sum()),
    minilm_top1_str_K10_archived=int(minilm_hit.sum()), stage1_top1_str=int(s1_hit.sum()),
    versions=dict(python=platform.python_version(), torch=torch.__version__, transformers=transformers.__version__,
                  sentence_transformers=sentence_transformers.__version__, numpy=np.__version__, pandas=pd.__version__),
    device=device, machine=platform.machine())
(OUT / "canine_run_manifest.json").write_text(json.dumps(manifest, indent=2))
con.close()
print(json.dumps({k: manifest[k] for k in ("timestamp", "backbone", "k_report_canine", "tau_canine", "canine_top1_str_K10", "minilm_top1_str_K10_archived", "stage1_top1_str")}, indent=1))
print(f"\nwrote {OUT/'canine_run_manifest.json'}; database closed")

{
 "timestamp": "2026-10-01 19:30:36",
 "backbone": "google/canine-c",
 "k_report_canine": 10,
 "tau_canine": 0.8400000000000002,
 "canine_top1_str_K10": 657,
 "minilm_top1_str_K10_archived": 653,
 "stage1_top1_str": 641
}

wrote output_canine/canine_run_manifest.json; database closed
